In [ ]:
import os
import re
import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
from shapely.geometry import mapping

folder = "/Users/aditiiyer/Documents/Thesis_prep/Geotiffs_monthly"
aoi_path = "/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/wetland_continuous_outline.gpkg"

aoi = gpd.read_file(aoi_path)
pattern = re.compile(r"TsoKar_(\d{4})_(\d{2})")

for f in os.listdir(folder):
    if not f.endswith(".tif"):
        continue

    print("Checking:", f)

    tif = os.path.join(folder, f)

    with rasterio.open(tif) as src:
        print("Raster CRS:", src.crs)
        print("Raster bounds:", src.bounds)

        aoi_r = aoi.to_crs(src.crs)
        print("AOI bounds in raster CRS:", aoi_r.total_bounds)

        try:
            clipped, transform = rasterio.mask.mask(src, [mapping(aoi_r.unary_union)], crop=True)
        except Exception as e:
            print("Clip error:", e)
            continue

        arr = clipped
        print("Clipped shape:", arr.shape)

        unique_vals = np.unique(arr)
        print("Unique values:", unique_vals[:20])
        print("Min:", arr.min(), "Max:", arr.max())

        if np.all(arr == 0):
            print("Result is all zeros.")
        else:
            print("Nonzero values present.")

    print("----")


In [2]:
import os
import re
import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
from shapely.geometry import mapping
from pyproj import Transformer

folder = "/Users/aditiiyer/Documents/Thesis_prep/Geotiffs_monthly"
aoi_path = "/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/wetland_continuous_outline.gpkg"

aoi = gpd.read_file(aoi_path)
pattern = re.compile(r"TsoKar_(\d{4})_(\d{2})")

records = []

for f in os.listdir(folder):
    if not f.endswith(".tif"):
        continue

    m = pattern.search(f)
    if not m:
        continue

    year = int(m.group(1))
    month = int(m.group(2))

    tif_path = os.path.join(folder, f)

    with rasterio.open(tif_path) as src:
        aoi_r = aoi.to_crs(src.crs)
        clipped, transform = rasterio.mask.mask(
            src,
            [mapping(aoi_r.union_all())],
            crop=True,
            filled=False
        )

        mask = np.any(clipped.mask, axis=0)
        valid = ~mask
        ys, xs = np.where(valid)

        xs_world, ys_world = rasterio.transform.xy(transform, ys, xs)
        xs_world = np.array(xs_world)
        ys_world = np.array(ys_world)

        transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(xs_world, ys_world)

        rows = []
        for i in range(clipped.shape[0]):
            vals = clipped[i].data[ys, xs]
            rows.append(vals)
        rows = np.vstack(rows).T

        df = pd.DataFrame(rows, columns=[f"band_{i+1}" for i in range(clipped.shape[0])])
        df["x"] = xs_world
        df["y"] = ys_world
        df["lon"] = lon
        df["lat"] = lat
        df["year"] = year
        df["month"] = month

        records.append(df)

full_df = pd.concat(records, ignore_index=True)

In [3]:
full_df.shape

(8684324, 40)

In [4]:
full_df.sample(10)

,band_1,band_2,band_3,band_4,band_5,band_6,band_7,band_8,band_9,band_10,...,band_31,band_32,band_33,band_34,x,y,lon,lat,year,month
6477788,0.06010,0.09310,0.12800,0.13220,0.16630,0.21140,0.22905,0.28170,0.24325,0.14090,...,0.250048,-0.375144,278.325500,1.858009,223945.0,3684515.0,78.036539,33.264383,2023,7
3359159,0.09055,0.09575,0.13295,0.19065,0.20350,0.22205,0.24585,0.28685,0.30120,0.27305,...,0.006647,-0.385772,260.319519,0.370015,222135.0,3695405.0,78.013786,33.362012,2024,4
7907980,0.11460,0.09230,0.12075,0.11200,0.11660,0.07225,0.07725,0.06060,0.06245,0.14345,...,-0.048147,0.331680,268.250458,0.134773,223365.0,3693115.0,78.027691,33.341701,2025,6
5819800,0.18460,0.23730,0.29190,0.21990,0.19865,0.07655,0.06210,0.05265,0.02415,0.00180,...,0.907360,0.689065,267.564423,0.132918,218845.0,3689565.0,77.980288,33.308554,2023,10
3107814,0.38845,0.42860,0.45760,0.46460,0.48470,0.46445,0.47285,0.46620,0.46485,0.38245,...,-0.019628,-0.012704,277.396118,0.160053,223155.0,3687955.0,78.027017,33.295167,2023,9
2444695,0.48630,0.54040,0.52670,0.50400,0.50870,0.47770,0.45850,0.45950,0.42410,0.30970,...,0.295963,0.068140,255.904831,0.792245,222075.0,3690045.0,78.014791,33.313715,2024,3
8178735,0.28940,0.31320,0.32760,0.33960,0.36790,0.37030,0.38430,0.36560,0.38650,0.33680,...,0.337596,-0.054818,281.457184,1.324529,222345.0,3689835.0,78.017752,33.311893,2025,7
654185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,250.765442,0.413826,224195.0,3690375.0,78.037435,33.317233,2022,2
4913189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,256.821289,0.270836,224385.0,3683145.0,78.041673,33.252154,2023,3
1013411,0.36425,0.63180,0.62120,0.62700,0.57660,0.56395,0.56895,0.62020,0.54770,0.34495,...,0.257416,0.003983,276.068268,0.473856,217275.0,3691375.0,77.962880,33.324446,2022,6


In [5]:
rename_map = {
    "band_1": "B1",
    "band_2": "B2",
    "band_3": "B3",
    "band_4": "B4",
    "band_5": "B5",
    "band_6": "B6",
    "band_7": "B7",
    "band_8": "B8",
    "band_9": "B8A",
    "band_10": "B9",
    "band_11": "B11",
    "band_12": "B12",
    "band_13": "AOT",
    "band_14": "WVP",
    "band_15": "SCL",
    "band_16": "TCI_R",
    "band_17": "TCI_G",
    "band_18": "TCI_B",
    "band_19": "MSK_CLDPRB",
    "band_20": "MSK_SNWPRB",
    "band_21": "QA10",
    "band_22": "QA20",
    "band_23": "QA60",
    "band_24": "MSK_CLASSI_OPAQUE",
    "band_25": "MSK_CLASSI_CIRRUS",
    "band_26": "MSK_CLASSI_SNOW_ICE",
    "band_27": "NDVI",
    "band_28": "EVI",
    "band_29": "SAVI",
    "band_30": "GCC",
    "band_31": "NDMI",
    "band_32": "NDWI",
    "band_33": "T2M",
    "band_34": "TP"
}
full_df = full_df.rename(columns=rename_map)
full_df.sample(10)

,B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,...,NDMI,NDWI,T2M,TP,x,y,lon,lat,year,month
6508975,0.00485,0.0081,0.01465,0.00870,0.01520,0.00555,0.00615,0.00570,0.00295,0.00155,...,-0.008426,0.464722,278.323181,1.857317,224965.0,3682145.0,78.048194,33.243294,2023,7
4322438,0.02220,0.0275,0.03335,0.03085,0.04115,0.02435,0.02580,0.02195,0.01455,0.00845,...,0.282840,0.153981,275.119812,0.395949,224625.0,3683045.0,78.044277,33.251315,2024,6
6598222,0.18510,0.2085,0.24950,0.19660,0.18055,0.07630,0.07715,0.06150,0.03755,0.00405,...,0.874608,0.604502,273.020142,0.409462,220205.0,3689865.0,77.994785,33.311610,2023,6
6039728,0.24160,0.2686,0.31000,0.26040,0.23590,0.14880,0.13470,0.12180,0.10150,0.05790,...,0.880325,0.341136,263.228455,0.099861,221425.0,3688955.0,78.008153,33.303729,2023,11
2260041,0.19820,0.1995,0.19990,0.19070,0.18775,0.17470,0.16595,0.15660,0.14920,0.12390,...,0.484523,0.121459,250.805374,0.712027,222105.0,3689775.0,78.015196,33.311291,2024,2
1828721,0.12995,0.1747,0.21585,0.17500,0.16980,0.09410,0.09320,0.07835,0.05195,0.01460,...,0.842849,0.456286,264.137390,0.207008,223735.0,3690645.0,78.032417,33.319547,2022,10
3984126,0.07945,0.0628,0.09835,0.16490,0.19445,0.21915,0.24025,0.26245,0.27315,0.23520,...,0.323514,-0.454098,255.955475,0.236800,224175.0,3691075.0,78.037006,33.323533,2024,12
6718972,0.20635,0.2140,0.25420,0.28910,0.29245,0.29465,0.30265,0.31660,0.30670,0.28685,...,0.125315,-0.116165,257.969238,0.070857,221665.0,3694625.0,78.008981,33.354864,2023,12
585016,0.09980,0.1656,0.22820,0.26030,0.23330,0.22860,0.23640,0.27840,0.24470,0.22410,...,0.021757,-0.119822,258.674164,0.130929,223885.0,3682295.0,78.036571,33.244370,2022,3
2753370,0.32290,0.1593,0.19770,0.27380,0.30950,0.32240,0.34100,0.36580,0.36855,0.41430,...,0.162670,-0.314828,256.621521,0.165447,225805.0,3682455.0,78.057106,33.246300,2024,1


In [6]:
full_df = full_df.drop(['x', 'y','AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB',
       'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE',
       'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE'], axis = 1)

In [7]:
full_df.sample(10)

,B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,...,SAVI,GCC,NDMI,NDWI,T2M,TP,lon,lat,year,month
2949083,0.03530,0.04625,0.07720,0.0656,0.09535,0.13025,0.14170,0.12000,0.13730,0.09015,...,0.131925,0.411848,0.341325,-0.242901,280.117218,0.519173,78.039175,33.246686,2023,8
1162871,0.10760,0.06545,0.10260,0.1136,0.15165,0.14240,0.15520,0.15570,0.14215,0.19760,...,0.060003,0.365624,0.070890,-0.172013,276.124634,0.439370,78.056566,33.252779,2022,6
3640800,0.30655,0.42945,0.45860,0.4582,0.47245,0.46615,0.47275,0.45985,0.46330,0.31590,...,-0.000055,0.339297,0.015836,-0.005810,268.201843,0.139628,78.012078,33.311396,2024,5
1736282,0.06360,0.08000,0.09555,0.1010,0.13280,0.15345,0.14975,0.16305,0.14670,0.08590,...,0.121703,0.345539,0.689565,-0.271193,255.861298,0.328020,78.033580,33.269361,2022,11
5930639,0.12735,0.17310,0.22870,0.2622,0.28060,0.28695,0.30250,0.30270,0.31165,0.28855,...,0.054623,0.341165,0.024450,-0.136891,263.330780,0.098570,78.019693,33.352598,2023,11
819912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,252.609634,0.215441,78.026007,33.324893,2022,12
6197455,0.27855,0.33030,0.35940,0.3904,0.38295,0.38230,0.39165,0.40430,0.39770,0.30245,...,0.023739,0.333314,0.077600,-0.067245,265.158630,0.479505,77.952802,33.311668,2023,5
4819729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,256.840515,0.263250,78.029314,33.312892,2023,3
4005764,0.32885,0.34560,0.37215,0.3724,0.39860,0.40180,0.39425,0.38280,0.39510,0.32275,...,0.016074,0.340156,0.062401,-0.021281,255.858978,0.246643,77.990841,33.317014,2024,12
6248536,0.18220,0.23300,0.26540,0.2350,0.22610,0.16230,0.16220,0.14440,0.12560,0.08320,...,-0.154537,0.361876,0.338739,0.295266,265.228912,0.454581,78.030766,33.301748,2023,5


In [8]:
full_df["date"] = pd.to_datetime(
    full_df["year"].astype(str) + "-" + full_df["month"].astype(str) + "-01"
)

In [9]:
full_df.to_csv("/Users/aditiiyer/Documents/Thesis/Appendices/raw_data.csv")

In [10]:
import pandas as pd

In [11]:
full_df = pd.read_csv("/Users/aditiiyer/Documents/Thesis/Appendices/raw_data.csv")

In [12]:
# ====== CONFIG ======
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Columns: lon, lat, date (datetime64), and features below
FEATURES_BANDS   = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
FEATURES_INDICES = ["NDVI","EVI","SAVI","GCC","NDMI","NDWI"]
FEATURES_ALL     = FEATURES_BANDS + FEATURES_INDICES

OUTPUT_DIR = "/Users/aditiiyer/Documents/Thesis/Report_Figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ensure date is datetime (YYYY-MM-01 from your cleaning step)
full_df["date"] = pd.to_datetime(full_df["date"]).dt.to_period("M").dt.to_timestamp()

# ====== 1) MONTH COVERAGE ≥ x% ======
# "Valid" means the row has all 16 features non-null for that month
row_ok = full_df[FEATURES_ALL].notna().all(axis=1)

# If you want "any feature present" instead, use:
# row_ok = df_tidy[FEATURES_ALL].notna().any(axis=1)

coverage = (
    full_df.assign(valid=row_ok)
           .groupby("date", as_index=False)
           .agg(total_pixels=("valid","size"), valid_pixels=("valid","sum"))
)
coverage["valid_frac"] = coverage["valid_pixels"] / coverage["total_pixels"]
good_months = coverage.loc[coverage["valid_frac"] >= 0.80, "date"].sort_values().tolist()

# Save coverage table
coverage.to_csv(os.path.join(OUTPUT_DIR, "monthly_valid_coverage.csv"), index=False)

print("Months with ≥80% valid pixels:")
for d in good_months:
    print(d.strftime("%Y-%m"))

# ====== 2) COVARIANCE HEATMAPS (SEABORN) ======
# Use only rows that are fully valid across all features (consistent sample)
valid_df = full_df.loc[row_ok, FEATURES_ALL].copy()

# Compute covariance matrices
corr_bands   = valid_df[FEATURES_BANDS].corr(min_periods=1)
corr_indices = valid_df[FEATURES_INDICES].corr(min_periods=1)

# Pretty heatmap helper
def plot_heatmap(mat, title, fname):
    plt.figure(figsize=(8, 6), dpi=300)
    mask = np.triu(np.ones_like(mat, dtype=bool))  # show lower triangle for clarity
    ax = sns.heatmap(
        mat, mask=mask, annot=True, fmt=".2f",
        cmap="vlag", center=0, square=True, cbar_kws={"shrink": 0.8}
    )
    ax.set_title(title, pad=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=300, bbox_inches="tight")
    plt.close()

plot_heatmap(corr_bands,   "Correlation (Sentinel-2 Bands)",   "corr_bands.png")
plot_heatmap(corr_indices, "Correlation (Vegetation/Moisture Indices)", "corr_indices.png")

# ====== 3) TIME SERIES FOR INDICES ======
# We’ll compute monthly median and IQR (25–75%) across pixels for each index,
# then plot all six indices together (clean, report-ready). You’ll get:
#  - one multi-line plot (medians)
#  - one ribbon plot (median with IQR band) per index (optional, nice for appendices)

# Aggregate monthly stats
agg_funcs = {}
for idx in FEATURES_INDICES:
    agg_funcs[idx+"_median"] = (idx, "median")
    agg_funcs[idx+"_q25"]    = (idx, lambda x: x.quantile(0.25))
    agg_funcs[idx+"_q75"]    = (idx, lambda x: x.quantile(0.75))

monthly_idx = (
    full_df.groupby("date", as_index=False)
           .agg(**agg_funcs)
           .sort_values("date")
)

# ----- 3a) Multi-line time series (medians) -----
plt.figure(figsize=(10, 4.5), dpi=300)
for idx in FEATURES_INDICES:
    plt.plot(
        monthly_idx["date"], monthly_idx[f"{idx}_median"],
        linewidth=1.8, label=idx
    )
plt.xlabel("Month")
plt.ylabel("Index value (median)")
plt.title("Monthly Indices — Median Across Pixels")
plt.xticks(rotation=0)
plt.grid(alpha=0.3, linewidth=0.6)
plt.legend(ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "indices_timeseries_median.png"), dpi=300, bbox_inches="tight")
plt.close()

# ----- 3b) Optional small-multiple ribbons (median + IQR) -----
# Comment out this section if you only want a single figure.
n = len(FEATURES_INDICES)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 6), dpi=300, sharex=True)
axes = axes.flatten()

for i, idx in enumerate(FEATURES_INDICES):
    ax = axes[i]
    x  = monthly_idx["date"].values
    med = monthly_idx[f"{idx}_median"].values
    q25 = monthly_idx[f"{idx}_q25"].values
    q75 = monthly_idx[f"{idx}_q75"].values

    ax.plot(x, med, linewidth=1.8)
    ax.fill_between(x, q25, q75, alpha=0.25)
    ax.set_title(idx)
    ax.grid(alpha=0.3, linewidth=0.6)

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

for ax in axes:
    for label in ax.get_xticklabels():
        label.set_fontsize(6)
        label.set_rotation(45)
        label.set_ha("right")

fig.supylabel("Index value")
fig.supxlabel("Month")
fig.suptitle("Monthly Indices — Median and IQR Across Pixels (SG)", y=0.99)
fig.tight_layout(rect=[0, 0.00, 1, 0.97])
fig.savefig(os.path.join(OUTPUT_DIR, "indices_timeseries_small_multiples.png"), dpi=300, bbox_inches="tight")
plt.close()

# ====== 4) (Optional) Coverage bar chart for report ======
plt.figure(figsize=(10, 3.8), dpi=300)
plt.bar(coverage["date"].dt.strftime("%Y-%m"), coverage["valid_frac"], width=0.8)
plt.axhline(0.70, linestyle="--", linewidth=1.2)
plt.xticks(rotation=90)
plt.ylim(0, 1.0)
plt.ylabel("Valid fraction")
plt.title("Monthly Coverage (fraction of pixels with complete data)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "monthly_coverage_bar.png"), dpi=300, bbox_inches="tight")
plt.close()


Months with ≥80% valid pixels:
2022-03
2022-04
2022-05
2022-06
2022-07
2022-08
2022-09
2022-10
2023-04
2023-05
2023-06
2023-07
2023-08
2023-09
2023-10
2023-11
2024-04
2024-05
2024-06
2024-07
2024-08
2024-09
2024-10
2025-04
2025-05
2025-06
2025-07
2025-08


In [13]:
# ============================================
# - per pixel loops
# - Tiles by lon/lat to keep working set small
# - Float32 arrays to halve memory
# - Streams output to Parquet with pyarrow
# - SG smoothing for all features
# - 1/R and log(1/R) for BANDS ONLY (positive reflectance)
# - First derivative for all features
# ============================================

import os
import math
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
import pyarrow as pa
import pyarrow.parquet as pq

# ----------------------------
# CONFIG
# ----------------------------
FEATURES_BANDS   = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
FEATURES_INDICES = ["NDVI","EVI","SAVI","GCC","NDMI","NDWI"]
FEATURES_ALL     = FEATURES_BANDS + FEATURES_INDICES

# Savitzky–Golay params (adaptive per series if short)
SG_WINDOW_DEFAULT = 7   # odd; will auto-reduce if needed
SG_POLY_DEFAULT   = 2
DERIV_DELTA       = 1.0

# Treat very small/invalid reflectance as NaN before 1/R, log(1/R)
REFLECTANCE_FLOOR = 1e-4

# Tiling (in degrees). 0.02 ≈ ~2 km in lat; adjust for your AOI.
LON_BIN = 0.02
LAT_BIN = 0.02

# Output
OUT_DIR = "processed_streaming"
OUT_PARQUET = os.path.join(OUT_DIR, "preprocessed_soc.parquet")
os.makedirs(OUT_DIR, exist_ok=True)

# Columns to output: keep it minimal to save space
KEEP_ORIGINALS = False  # set True to retain original features in output

# ----------------------------
# PREP INPUT
# ----------------------------
# Normalize month format to month-start timestamps
good_months = pd.to_datetime(pd.Series(good_months)).dt.to_period("M").dt.to_timestamp()

# Keep only rows where date is in the good months list
df_pp = full_df[full_df["date"].isin(good_months)].copy()

# Ensure proper types & monthly timestamps
df_pp["date"] = pd.to_datetime(df_pp["date"]).dt.to_period("M").dt.to_timestamp()
# Cast features to float32 (big RAM saver)
for c in FEATURES_ALL:
    df_pp[c] = df_pp[c].astype("float32")

# Build tile keys (integers) so each tile is ~LON_BIN × LAT_BIN
lon0, lat0 = float(df_pp["lon"].min()), float(df_pp["lat"].min())
df_pp["lon_tile"] = ((df_pp["lon"] - lon0) / LON_BIN).astype(int)
df_pp["lat_tile"] = ((df_pp["lat"] - lat0) / LAT_BIN).astype(int)

# ----------------------------
# HELPERS
# ----------------------------
def _adaptive_sg_params(n, win_default=SG_WINDOW_DEFAULT, poly_default=SG_POLY_DEFAULT):
    if n < 3:
        return None, None
    w = min(win_default, n if n % 2 == 1 else n - 1)
    if w < 3:
        w = 3 if n >= 3 else None
    if w is None: 
        return None, None
    p = min(poly_default, w - 1)
    return w, p

def _sg_pair(series):
    """Return (smoothed, first_derivative) for a 1D time series with NaNs handled."""
    s = series.astype("float32").copy()
    # Interpolate internal gaps (time-aware)
    s = pd.Series(s, index=series.index).interpolate(method="time", limit_direction="both").values.astype("float32")
    n = len(s)
    w, p = _adaptive_sg_params(n)
    if w is None:
        return series.values.astype("float32"), np.full(n, np.nan, dtype="float32")
    y_sg = savgol_filter(s, w, p, deriv=0, delta=DERIV_DELTA, mode="interp").astype("float32")
    y_d1 = savgol_filter(s, w, p, deriv=1, delta=DERIV_DELTA, mode="interp").astype("float32")
    return y_sg, y_d1

def _process_pixel(pdf):
    """Process all features for a single pixel (lon,lat), indexed by date."""
    out = {"date": pdf.index.values.astype("datetime64[ns]")}
    # SG + derivative for all features
    for f in FEATURES_ALL:
        y_sg, y_d1 = _sg_pair(pdf[f])
        out[f"{f}_sg"] = y_sg
        out[f"{f}_d1"] = y_d1
    # 1/R and log(1/R) for BANDS only
    for f in FEATURES_BANDS:
        y = out[f"{f}_sg"]  # use smoothed reflectance
        y = np.where(np.isfinite(y) & (y > REFLECTANCE_FLOOR), y, np.nan).astype("float32")
        inv = np.divide(1.0, y, out=np.full_like(y, np.nan), where=np.isfinite(y))
        loginv = np.where(np.isfinite(inv) & (inv > 0), np.log(inv), np.nan).astype("float32")
        out[f"{f}_inv"] = inv
        out[f"{f}_loginv"] = loginv
    return out

def _tile_iterator(df_tiles):
    """Yield (tile_key, tile_df) lazily to keep RAM low."""
    for (tx, ty), sub in df_tiles.groupby(["lon_tile", "lat_tile"], sort=False):
        yield (int(tx), int(ty)), sub

# ----------------------------
# PARQUET WRITER (streaming)
# ----------------------------
writer = None

# Order dates once (speeds up per-pixel sort)
all_dates = np.sort(df_pp["date"].unique())

for (tx, ty), tile in _tile_iterator(df_pp):
    # Work only with necessary columns
    cols = ["lon","lat","date"] + FEATURES_ALL
    tile = tile[cols].copy()

    # Sort by pixel/time and convert to a MultiIndex
    tile = tile.sort_values(["lon","lat","date"])
    tile = tile.set_index(["lon","lat","date"])

    # Process pixel by pixel (generator style)
    rows_out = []
    # groupby at this scale is fine (small tile); keeps memory reasonable
    for (lon, lat), g in tile.groupby(level=[0,1], sort=False):
        g = g.reset_index(level=[0,1], drop=True)
        # reindex to full monthly timeline so interpolation is clean
        g = g.reindex(all_dates, copy=False)
        out = _process_pixel(g[FEATURES_ALL])

        # pack into DataFrame; keep only needed columns to save memory
        df_pp_out = pd.DataFrame(out)
        df_pp_out.insert(0, "lat", lat)
        df_pp_out.insert(0, "lon", lon)
        if KEEP_ORIGINALS:
            for f in FEATURES_ALL:
                df_pp_out[f] = g[f].values.astype("float32")
        rows_out.append(df_pp_out)

        # Flush periodically to limit RAM
        if len(rows_out) >= 200:  # tune this chunk size to your RAM
            batch = pd.concat(rows_out, ignore_index=True)
            table = pa.Table.from_pandas(batch, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(OUT_PARQUET, table.schema, compression="zstd")
            writer.write_table(table)
            rows_out.clear()

    # flush remaining rows for this tile
    if rows_out:
        batch = pd.concat(rows_out, ignore_index=True)
        table = pa.Table.from_pandas(batch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(OUT_PARQUET, table.schema, compression="zstd")
        writer.write_table(table)

# close writer
if writer is not None:
    writer.close()

print(f"✅ Streamed to {OUT_PARQUET}")

✅ Streamed to processed_streaming/preprocessed_soc.parquet


In [14]:
import pandas as pd

# Paths to your Parquet(s)
INPUT_FILE = "/Users/aditiiyer/Documents/Thesis/processed_streaming/preprocessed_soc.parquet"

# List only the columns you want
FEATURES_BANDS_SG   = [f"{b}_sg" for b in ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]]
FEATURES_INDICES_SG = [f"{i}_sg" for i in ["NDVI","EVI","SAVI","GCC","NDMI","NDWI"]]
CLIMATE_VARS = [f"{i}_sg" for i in ["T2M", "TP"]]

COLS_TO_LOAD = ["lon", "lat", "date"] + FEATURES_INDICES_SG + FEATURES_BANDS_SG

# Load only those columns
df_sg = pd.read_parquet(INPUT_FILE, columns=COLS_TO_LOAD)

print(df_sg.memory_usage(deep=True).sum() / (1024**2), "MB in memory")

463.79306411743164 MB in memory


In [15]:
df_sg.head()

,lon,lat,date,NDVI_sg,EVI_sg,SAVI_sg,GCC_sg,NDMI_sg,NDWI_sg,B2_sg,B3_sg,B4_sg,B5_sg,B6_sg,B7_sg,B8_sg,B8A_sg,B11_sg,B12_sg
0,78.000496,33.364215,2022-03-01,0.042582,0.037589,0.037401,0.331567,-0.025571,-0.140641,0.217086,0.271865,0.326311,0.344725,0.342212,0.349562,0.354844,0.364077,0.384855,0.310714
1,78.000496,33.364215,2022-04-01,0.100294,0.105820,0.086497,0.332886,0.053149,-0.172043,0.204739,0.253118,0.295050,0.332886,0.351836,0.362039,0.355064,0.372404,0.332218,0.257425
2,78.000496,33.364215,2022-05-01,0.159409,0.163929,0.132331,0.336393,0.111417,-0.210769,0.187775,0.231614,0.262696,0.314632,0.352386,0.365604,0.352039,0.373871,0.288796,0.213368
3,78.000496,33.364215,2022-06-01,0.219928,0.211916,0.174901,0.342090,0.149233,-0.256819,0.166193,0.207355,0.229250,0.289964,0.343862,0.360255,0.345769,0.368481,0.254590,0.178543
4,78.000496,33.364215,2022-07-01,0.324752,0.284639,0.244142,0.357353,0.181952,-0.334716,0.133426,0.175048,0.181333,0.251679,0.330814,0.353762,0.342838,0.363298,0.224307,0.146048


In [16]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
OUTPUT_DIR = "/Users/aditiiyer/Documents/Thesis/Report_Figures_postprocessing"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define SG column lists
FEATURES_BANDS   = [f"{b}_sg" for b in ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]]
FEATURES_INDICES = [f"{i}_sg" for i in ["NDVI","EVI","SAVI","GCC","NDMI","NDWI"]]
FEATURES_ALL     = FEATURES_BANDS + FEATURES_INDICES

# =========================
# LOAD
# =========================
df_proc = df_sg.copy()
df_proc["date"] = pd.to_datetime(df_proc["date"]).dt.to_period("M").dt.to_timestamp()

# =========================
# 1) MONTH COVERAGE ≥ 80%
# =========================
row_ok = df_proc[FEATURES_ALL].notna().all(axis=1)

coverage = (
    df_proc.assign(valid=row_ok)
           .groupby("date", as_index=False)
           .agg(total_pixels=("valid","size"), valid_pixels=("valid","sum"))
)
coverage["valid_frac"] = coverage["valid_pixels"] / coverage["total_pixels"]
good_months = coverage.loc[coverage["valid_frac"] >= 0.80, "date"].sort_values().tolist()

coverage.to_csv(os.path.join(OUTPUT_DIR, "monthly_valid_coverage.csv"), index=False)

print("Months with ≥80% valid pixels:")
for d in good_months:
    print(d.strftime("%Y-%m"))

# =========================
# 2) COVARIANCE HEATMAPS
# =========================
valid_df = df_proc.loc[row_ok, FEATURES_ALL].copy()

corr_bands   = valid_df[FEATURES_BANDS].corr(min_periods=1)
corr_indices = valid_df[FEATURES_INDICES].corr(min_periods=1)

def plot_heatmap(mat, title, fname):
    plt.figure(figsize=(8, 6), dpi=300)
    mask = np.triu(np.ones_like(mat, dtype=bool))
    ax = sns.heatmap(
        mat, mask=mask, annot=True, fmt=".2f",
        cmap="vlag", center=0, square=True, cbar_kws={"shrink": 0.8}
    )
    ax.set_title(title, pad=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=300, bbox_inches="tight")
    plt.close()

plot_heatmap(corr_bands,   "Correlation (Sentinel-2 Bands, SG)",   "corr_bands_sg.png")
plot_heatmap(corr_indices, "Correlation (Indices, SG)", "corr_indices_sg.png")

# =========================
# 3) TIME SERIES FOR INDICES
# =========================
agg_funcs = {}
for idx in FEATURES_INDICES:
    agg_funcs[idx+"_median"] = (idx, "median")
    agg_funcs[idx+"_q25"]    = (idx, lambda x: x.quantile(0.25))
    agg_funcs[idx+"_q75"]    = (idx, lambda x: x.quantile(0.75))

monthly_idx = (
    df_proc.groupby("date", as_index=False)
           .agg(**agg_funcs)
           .sort_values("date")
)

# ----- Multi-line medians -----
plt.figure(figsize=(10, 4.5), dpi=300)
for idx in FEATURES_INDICES:
    plt.plot(
        monthly_idx["date"], monthly_idx[f"{idx}_median"],
        linewidth=1.8, label=idx.replace("_sg", "")
    )
plt.xlabel("Month")
plt.ylabel("Index value (median, SG-smoothed)")
plt.title("Monthly Indices — Median Across Pixels (SG)")
plt.xticks(rotation=0)
plt.grid(alpha=0.3, linewidth=0.6)
plt.legend(ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "indices_timeseries_median_sg.png"), dpi=300, bbox_inches="tight")
plt.close()

# ----- Small multiples with IQR ribbons -----
n = len(FEATURES_INDICES)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 6), dpi=300, sharex=True)
axes = axes.flatten()

for i, idx in enumerate(FEATURES_INDICES):
    ax = axes[i]
    x  = monthly_idx["date"].values
    med = monthly_idx[f"{idx}_median"].values
    q25 = monthly_idx[f"{idx}_q25"].values
    q75 = monthly_idx[f"{idx}_q75"].values

    ax.plot(x, med, linewidth=1.8)
    ax.fill_between(x, q25, q75, alpha=0.25)
    ax.set_title(idx.replace("_sg", ""))
    ax.grid(alpha=0.3, linewidth=0.6)

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

for ax in axes:
    for label in ax.get_xticklabels():
        label.set_fontsize(6)
        label.set_rotation(45)
        label.set_ha("right")

fig.supylabel("Index value (SG-smoothed)")
fig.supxlabel("Month")
fig.suptitle("Monthly Indices — Median and IQR Across Pixels (SG)", y=0.99)
fig.tight_layout(rect=[0, 0.00, 1, 0.97])
fig.savefig(os.path.join(OUTPUT_DIR, "indices_timeseries_small_multiples_sg.png"), dpi=300, bbox_inches="tight")
plt.close()


# =========================
# 4) COVERAGE BAR CHART
# =========================
plt.figure(figsize=(10, 3.8), dpi=300)
plt.bar(coverage["date"].dt.strftime("%Y-%m"), coverage["valid_frac"], width=0.8)
plt.axhline(0.80, linestyle="--", linewidth=1.2)
plt.xticks(rotation=90)
plt.ylim(0, 1.0)
plt.ylabel("Valid fraction")
plt.title("Monthly Coverage (fraction of pixels with complete SG data)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "monthly_coverage_bar_sg.png"), dpi=300, bbox_inches="tight")
plt.close()


Months with ≥80% valid pixels:
2022-03
2022-04
2022-05
2022-06
2022-07
2022-08
2022-09
2022-10
2023-04
2023-05
2023-06
2023-07
2023-08
2023-09
2023-10
2023-11
2024-04
2024-05
2024-06
2024-07
2024-08
2024-09
2024-10
2025-04
2025-05
2025-06
2025-07
2025-08


In [14]:
df_sg.to_csv("/Users/aditiiyer/Documents/Thesis/data_sg_filter.csv")

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- CONFIG ---
OUTPUT_DIR = "/Users/aditiiyer/Documents/Thesis/Extent_EDA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Thresholds (tune for your AOI)
NDVI_WETLAND_THRESHOLD = 0.3
NDWI_WATER_THRESHOLD   = 0.3

# Ensure correct date type
df = df_sg
df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()

# --- WETLAND EXTENT (NDVI-based) ---
df["wetland_flag"] = df["NDVI_sg"] > NDVI_WETLAND_THRESHOLD
wetland_monthly = (
    df.groupby("date", as_index=False)
      .agg(total_pixels=("wetland_flag", "size"),
           wetland_pixels=("wetland_flag", "sum"))
)
wetland_monthly["wetland_frac"] = wetland_monthly["wetland_pixels"] / wetland_monthly["total_pixels"]

# --- LAKE EXTENT (NDWI-based) ---
df["lake_flag"] = df["NDWI_sg"] > NDWI_WATER_THRESHOLD
lake_monthly = (
    df.groupby("date", as_index=False)
      .agg(total_pixels=("lake_flag", "size"),
           lake_pixels=("lake_flag", "sum"))
)
lake_monthly["lake_frac"] = lake_monthly["lake_pixels"] / lake_monthly["total_pixels"]

# --- MERGE ---
extent_monthly = wetland_monthly[["date", "wetland_frac"]].merge(
    lake_monthly[["date", "lake_frac"]], on="date"
)

# --- SAVE CSV ---
extent_monthly.to_csv(os.path.join(OUTPUT_DIR, "wetland_lake_extent.csv"), index=False)

# --- PLOT ---
plt.figure(figsize=(10, 4.5), dpi=300)
plt.plot(extent_monthly["date"], extent_monthly["wetland_frac"], label="Wetland extent (NDVI)", linewidth=1.8)
plt.plot(extent_monthly["date"], extent_monthly["lake_frac"], label="Lake extent (NDWI)", linewidth=1.8)
plt.xlabel("Month")
plt.ylabel("Fraction of pixels")
plt.title("Monthly Wetland & Lake Extent (SG-smoothed)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "wetland_lake_extent_timeseries.png"), dpi=300, bbox_inches="tight")
plt.close()

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu

# --- CONFIG ---
OUTPUT_DIR = "/Users/aditiiyer/Documents/Thesis/Extent_EDA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ensure correct date type
df_sg = pd.read_csv("/Users/aditiiyer/Documents/Thesis/data_sg_filter.csv")
df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()

# --- STORAGE ---
monthly_results = []

# --- PROCESS MONTHLY ---
for month, g in df.groupby("date"):
    # Drop NaNs for threshold calculation
    ndvi_vals = g["NDVI_sg"].dropna().values
    ndwi_vals = g["NDWI_sg"].dropna().values
    
    if len(ndvi_vals) == 0 or len(ndwi_vals) == 0:
        continue  # skip months with no data
    
    # --- Otsu thresholds ---
    ndvi_thresh = threshold_otsu(ndvi_vals)
    ndwi_thresh = threshold_otsu(ndwi_vals)
    
    # --- Classification ---
    wetland_flag = g["NDVI_sg"] > ndvi_thresh
    lake_flag    = g["NDWI_sg"] > ndwi_thresh
    
    wetland_frac = wetland_flag.mean()
    lake_frac    = lake_flag.mean()
    
    monthly_results.append({
        "date": month,
        "ndvi_otsu_thresh": ndvi_thresh,
        "ndwi_otsu_thresh": ndwi_thresh,
        "wetland_frac": wetland_frac,
        "lake_frac": lake_frac
    })

# --- TO DATAFRAME ---
extent_monthly = pd.DataFrame(monthly_results).sort_values("date")

# --- SAVE CSV ---
extent_monthly.to_csv(os.path.join(OUTPUT_DIR, "wetland_lake_extent_otsu.csv"), index=False)

# --- PLOT ---
plt.figure(figsize=(10, 4.5), dpi=300)
plt.plot(extent_monthly["date"], extent_monthly["wetland_frac"], label="Wetland extent (NDVI Otsu)", linewidth=1.8)
plt.plot(extent_monthly["date"], extent_monthly["lake_frac"], label="Lake extent (NDWI Otsu)", linewidth=1.8)
plt.xlabel("Month")
plt.ylabel("Fraction of pixels")
plt.title("Monthly Wetland & Lake Extent — Otsu Threshold (SG-smoothed)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "wetland_lake_extent_timeseries_otsu.png"), dpi=300, bbox_inches="tight")
plt.close()

In [20]:
# ============================================
# Build multi-year wetland extent from 3 rasters
# - Inputs: annual Otsu masks (0/1), same grid/CRS
# - Outputs: union, intersection, majority(>=2/3), fractional persistence
# - Optional: polygonize majority to GeoPackage & report area
# ============================================

import os
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape as shapely_from_geojson
import geopandas as gpd

# ---------- CONFIG ----------
IN_2022 = "/Users/aditiiyer/Documents/Thesis/Otsu_extents_yoy/2022_Otsu_extent.tif"
IN_2023 = "/Users/aditiiyer/Documents/Thesis/Otsu_extents_yoy/2023_Otsu_extent.tif"
IN_2024 = "/Users/aditiiyer/Documents/Thesis/Otsu_extents_yoy/2024_Otsu_extent.tif"
IN_2025 = "/Users/aditiiyer/Documents/Thesis/Otsu_extents_yoy/2025_Otsu_extent.tif"

OUT_DIR = "wetland_extent_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_UNION      = os.path.join(OUT_DIR, "wetland_union.tif")
OUT_INTERSECT  = os.path.join(OUT_DIR, "wetland_intersection.tif")
OUT_HALF   = os.path.join(OUT_DIR, "wetland_majority_2of4.tif")
OUT_FRACTION   = os.path.join(OUT_DIR, "wetland_fraction_0to1.tif")
OUT_THIRD   = os.path.join(OUT_DIR, "wetland_majority_3of4.tif")
OUT_GPKG       = os.path.join(OUT_DIR, "wetland_majority_2of4.gpkg")  # polygonized

# If your input rasters lack nodata in metadata, set it here. Otherwise leave as None.
FORCE_NODATA = None  # e.g., 255

# Polygonize majority?
DO_POLYGONIZE = True
GPKG_LAYER    = "wetland_majority"

# ---------- HELPERS ----------
def open_rasters(paths):
    srcs = [rasterio.open(p) for p in paths]
    # sanity checks
    ref = srcs[0]
    for s in srcs[1:]:
        assert s.width == ref.width and s.height == ref.height, "Different raster sizes."
        assert s.crs == ref.crs, "Different CRS."
        assert s.transform == ref.transform, "Different geotransforms."
    return srcs

def write_like(ref, out_path, dtype, nodata, compress="LZW"):
    profile = ref.profile.copy()
    profile.update(
        dtype=dtype,
        count=1,
        nodata=nodata,
        compress=compress,
        bigtiff="IF_SAFER"
    )
    return rasterio.open(out_path, "w", **profile)

def window_iter(width, height, block=1024):
    for row in range(0, height, block):
        h = min(block, height - row)
        for col in range(0, width, block):
            w = min(block, width - col)
            yield Window(col_off=col, row_off=row, width=w, height=h)

# ---------- MAIN ----------
paths = [IN_2022, IN_2023, IN_2024, IN_2025]
srcs = open_rasters(paths)  # returns list of rasterio.DatasetReader objects
ref = srcs[0]

nodatas = [FORCE_NODATA if FORCE_NODATA is not None else s.nodata for s in srcs]
if any(nd is None for nd in nodatas):
    print("⚠️ Some inputs have nodata=None; treating all values as valid (0/1). Set FORCE_NODATA if needed.")
nd = nodatas[0] if all(n == nodatas[0] for n in nodatas) else None

# Prepare writers
out_union     = write_like(ref, OUT_UNION,     dtype="uint8",  nodata=0)
out_intersect = write_like(ref, OUT_INTERSECT, dtype="uint8",  nodata=0)
out_half  = write_like(ref, OUT_HALF,  dtype="uint8",  nodata=0)
out_third  = write_like(ref, OUT_THIRD,  dtype="uint8",  nodata=0)
out_fraction  = write_like(ref, OUT_FRACTION,  dtype="float32", nodata=np.nan)

try:
    for win in window_iter(ref.width, ref.height, block=1024):
        arrays = []
        valid_masks = []
        for s, s_nd in zip(srcs, nodatas):
            a = s.read(1, window=win).astype(np.float32)
            arrays.append(a)
            if s_nd is None:
                vm = np.ones_like(a, dtype=bool)
            else:
                vm = ~(a == s_nd)
            valid_masks.append(vm)

        valid_all = valid_masks[0] & valid_masks[1] & valid_masks[2] & valid_masks[3]

        a0 = np.where(valid_masks[0], arrays[0], 0.0)
        a1 = np.where(valid_masks[1], arrays[1], 0.0)
        a2 = np.where(valid_masks[2], arrays[2], 0.0)
        a3 = np.where(valid_masks[3], arrays[3], 0.0)

        a0 = (a0 > 0.5).astype(np.uint8)
        a1 = (a1 > 0.5).astype(np.uint8)
        a2 = (a2 > 0.5).astype(np.uint8)
        a3 = (a3 > 0.5).astype(np.uint8)

        ssum = a0 + a1 + a2 + a3

        union     = np.where(valid_all, (ssum >= 1).astype(np.uint8), 0)
        intersect = np.where(valid_all, (ssum == 4).astype(np.uint8), 0)
        half = np.where(valid_all, (ssum >= 2).astype(np.uint8), 0)
        third = np.where(valid_all, (ssum >= 3).astype(np.uint8), 0)
        fraction  = np.where(valid_all, (ssum / 4.0).astype(np.float32), np.nan)

        out_union.write(union, 1, window=win)
        out_intersect.write(intersect, 1, window=win)
        out_half.write(half, 1, window=win)
        out_third.write(third, 1, window=win)
        out_fraction.write(fraction, 1, window=win)
finally:
    out_union.close()
    out_intersect.close()
    out_half.close()
    out_third.close()
    out_fraction.close()
    for s in srcs:
        s.close()

print(f"✅ Wrote:\n  {OUT_UNION}\n  {OUT_INTERSECT}\n  {OUT_HALF}\n  {OUT_THIRD}\n  {OUT_FRACTION}")

# ---------- (Optional) Polygonize majority to GeoPackage ----------
if DO_POLYGONIZE:
    import rasterio
    from rasterio.features import shapes as rio_shapes

    with rasterio.open(OUT_THIRD) as src:
        transform = src.transform
        crs = src.crs
        geoms = []
        attrs = []

        # iterate in one go (safe if raster isn't massive; otherwise tile here too)
        arr = src.read(1)
        for geom, val in rio_shapes(arr.astype(np.uint8), transform=transform, connectivity=4):
            if int(val) == 1:
                geoms.append(shapely_from_geojson(geom))
                attrs.append({"class":"wetland_consensus_3of4"})

    gdf = gpd.GeoDataFrame(attrs, geometry=geoms, crs=crs)
    # Optional: dissolve to a single multipart polygon
    gdf = gdf.dissolve(by="class", as_index=False)

    if os.path.exists(OUT_GPKG):
        os.remove(OUT_GPKG)
    gdf.to_file(OUT_GPKG, layer="wetland_consensus", driver="GPKG")
    print(f"✅ Polygonized majority → {OUT_GPKG}")

⚠️ Some inputs have nodata=None; treating all values as valid (0/1). Set FORCE_NODATA if needed.
✅ Wrote:
  wetland_extent_outputs/wetland_union.tif
  wetland_extent_outputs/wetland_intersection.tif
  wetland_extent_outputs/wetland_majority_2of4.tif
  wetland_extent_outputs/wetland_majority_3of4.tif
  wetland_extent_outputs/wetland_fraction_0to1.tif
✅ Polygonized majority → wetland_extent_outputs/wetland_majority_2of4.gpkg


In [21]:
import rasterio
import numpy as np

with rasterio.open(OUT_UNION) as src_u, \
     rasterio.open(OUT_HALF) as src_2, \
     rasterio.open(OUT_THIRD) as src_3, \
     rasterio.open(OUT_INTERSECT) as src_4:

    union = src_u.read(1)
    arr2 = src_2.read(1)      # >=2 of 4 years
    arr3 = src_3.read(1)      # >=3 of 4 years
    arr4 = src_4.read(1)      # =4 of 4 years

    valid = (union == 1)
    total_valid = valid.sum()

    pct_2 = (arr2[valid].sum() / total_valid) * 100 if total_valid > 0 else 0
    pct_3 = (arr3[valid].sum() / total_valid) * 100 if total_valid > 0 else 0
    pct_4 = (arr4[valid].sum() / total_valid) * 100 if total_valid > 0 else 0

print("Persistence in at least 2 of 4 years, percent:", pct_2)
print("Persistence in at least 3 of 4 years, percent:", pct_3)
print("Persistence in all 4 years, percent:", pct_4)

Persistence in at least 2 of 4 years, percent: 74.84165845017463
Persistence in at least 3 of 4 years, percent: 36.86726048229416
Persistence in all 4 years, percent: 7.191380203546183


In [3]:
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio import features
from shapely.geometry import shape, Point

# --- Input files ---
tif_path = "/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/wetland_majority_3of4.tif"   # your raster extent
csv_path = "/Users/aditiiyer/Documents/Thesis/data_tsokar_2024_08.csv"            # your field data (lon, lat columns)

# --- Load raster and extract valid extent ---
with rasterio.open(tif_path) as src:
    raster_data = src.read(1)
    mask = raster_data > 0                 # adjust threshold if your extent is stored differently
    shapes = features.shapes(raster_data, mask=mask, transform=src.transform)
    polygons = [shape(geom) for geom, val in shapes if val > 0]
    wetland_gdf = gpd.GeoDataFrame(geometry=polygons, crs=src.crs)

# --- Load field points ---
df = pd.read_csv(csv_path, sep=";")
points_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitude"], df["Lattitude"]),
    crs="EPSG:4326"
)

# --- Reproject points to match raster CRS ---
points_gdf = points_gdf.to_crs(wetland_gdf.crs)

# --- Spatial join: check which points fall within wetland polygons ---
points_inside = gpd.sjoin(points_gdf, wetland_gdf, predicate="within", how="left")
points_inside["inside_wetland"] = ~points_inside["index_right"].isna()

# --- Summary ---
inside_count = points_inside["inside_wetland"].sum()
total_points = len(points_inside)
outside_count = total_points - inside_count

print(f"Total points: {total_points}")
print(f"Inside wetland: {inside_count}")
print(f"Outside wetland: {outside_count}")

# # --- Optional: Save outputs ---
# points_inside.to_file("field_points_within_extent.geojson", driver="GeoJSON")
# print("GeoJSON with inside/outside labels saved as field_points_within_extent.geojson")


Total points: 90
Inside wetland: 29
Outside wetland: 61


In [2]:
import geopandas as gpd
from shapely.ops import unary_union

def find_buffer_for_two_polygons(vector_path, start=0.0001, step=0.0001, max_buffer=0.01):
    """
    Iteratively buffers the geometry until exactly two polygons remain.
    
    Parameters
    ----------
    vector_path : str
        Path to input vector file (e.g., shapefile or geojson)
    start : float
        Initial buffer distance in coordinate units
    step : float
        Step size for increasing buffer
    max_buffer : float
        Maximum buffer distance to try
    
    Returns
    -------
    (float, GeoDataFrame)
        Tuple of the buffer distance and resulting polygons
    """
    gdf = gpd.read_file(vector_path)
    geom = unary_union(gdf.geometry)
    
    buffer_val = start
    result_gdf = None

    while buffer_val <= max_buffer:
        buffered = geom.buffer(buffer_val)
        buffered_gdf = gpd.GeoDataFrame(geometry=[buffered], crs=gdf.crs).explode(index_parts=False)
        n_polygons = len(buffered_gdf)
        
        if n_polygons == 2:
            result_gdf = buffered_gdf
            print(f"Found 2 polygons at buffer = {buffer_val}")
            break
        
        buffer_val += step

    if result_gdf is None:
        print("Did not reach exactly 2 polygons within max_buffer.")
    return buffer_val, result_gdf

vector_path = "/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/Vectorised_Intersection/Vectorised.shp"
buffer_val, two_polygons = find_buffer_for_two_polygons(vector_path)

if two_polygons is not None:
    two_polygons.to_file("wetland_two_polygons.gpkg", driver="GPKG")


Found 2 polygons at buffer = 0.0031999999999999984


In [15]:
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape
from shapely.ops import unary_union

path_tif = "/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/wetland_majority_3of4.tif"

with rasterio.open(path_tif) as src:
    data = src.read(1)
    transform = src.transform

# Vectorize wetland pixels
wetland_shapes = (
    {"properties": {"value": v}, "geometry": s}
    for s, v in shapes(data, mask=data == 1, transform=transform)
)
wetland_polys = [shape(f["geometry"]) for f in wetland_shapes]
wetland_union = unary_union(wetland_polys)

# Step 1: buffer out to merge all fragments
merged = wetland_union.buffer(0.0015)

# Step 2: buffer in slightly to restore approximate boundary thickness
continuous_outline = merged.buffer(-0.0011)
# Step 3: save
gdf_out = gpd.GeoDataFrame(geometry=[continuous_outline], crs=src.crs)
gdf_out.to_file("/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/wetland_continuous_outline.gpkg", driver="GPKG")


In [7]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

gpkg_path = "/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/wetland_continuous_outline.gpkg"
csv_path = "/Users/aditiiyer/Documents/Thesis/Combined_Data_24_25.csv"

wetland_gdf = gpd.read_file(gpkg_path)

df = pd.read_csv(csv_path)
points_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

points_gdf = points_gdf.to_crs(wetland_gdf.crs)

points_inside = gpd.sjoin(points_gdf, wetland_gdf, predicate="within", how="left")
points_inside["inside_wetland"] = ~points_inside["index_right"].isna()

inside_count = points_inside["inside_wetland"].sum()
total_points = len(points_inside)
outside_count = total_points - inside_count

print(f"Total points: {total_points}")
print(f"Inside wetland: {inside_count}")
print(f"Outside wetland: {outside_count}")

Total points: 119
Inside wetland: 80
Outside wetland: 39


In [8]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load CSV
ramsar_csv = "/Users/aditiiyer/Documents/Thesis/SG_indices_climate_vars_full_site.csv"
df = pd.read_csv(ramsar_csv)

# Convert to GeoDataFrame
gdf_ramsar = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)
gdf_extent = gpd.read_file("/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/wetland_continuous_outline.gpkg")

# Match CRS
if gdf_ramsar.crs != gdf_extent.crs:
    gdf_ramsar = gdf_ramsar.to_crs(gdf_extent.crs)

ramsar_clipped = gpd.sjoin(gdf_ramsar, gdf_extent, predicate="within", how="inner")

In [9]:
ramsar_clipped.to_csv("/Users/aditiiyer/Documents/Thesis/model_input.csv")